In [ ]:
# Import
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Bildpfad
IMAGE_PATH = "data/selected/easy/IMG_2474.png"

#IMAGE_PATH = "data/converted/IMG_2479.png"

In [ ]:
def classify_images(image_path):
    img_rgb = load_image(image_path)
    img_rgb = resize_image(img_rgb)
    img_filtered = apply_bilateral_filter(img_rgb)
    v_clahe = apply_clahe(img_filtered)
    img_hsv = convert_to_hsv(img_filtered)
    masks = create_color_masks(img_hsv)
    clean_masks = clean_color_masks(masks)
    found_signs = detect_signs(clean_masks)
    found_signs = filter_overlapping_signs(found_signs)
    found_signs = classify_found_signs(found_signs)

    # Farben für die Bounding Boxes
    color_map = {
        "rot":  (255, 0, 0),
        "blau": (0, 0, 255),
        "gelb": (255, 255, 0),
    }

    # Kopie
    img_result = img_rgb.copy()

    for sign in found_signs:
        x, y, w, h = sign["bbox"]
        color = color_map[sign["color"]]
        #BBox und Klasse zeichnen
        cv2.rectangle(img_result, (x, y), (x + w, y + h), color, 4)

        label = sign["klasse"]
        cv2.putText(img_result, label, (x, y - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        
    # Ergebnis anzeigen
    plt.figure(figsize=(14, 10))
    plt.imshow(img_result)
    plt.axis('off')
    plt.show()

In [ ]:
def load_image(image_path):
    img_bgr = cv2.imread(image_path)

    # bgr zu rgb 
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_rgb

---
Professorin Ivanovska nach Erlaubnis gebeten Bilder zu skalieren:  
  
Nicht besprocehn, aber weil die Bidler os groß sind sollte man diese auf ein einheitliches kleineres Bildforamt packen, damit alle Funktionen mit den entsprechenden Variablen gleichermaßen funktionieren und manceh Funktionen nicht mehr zu lange dauern


---

In [ ]:
def resize_image(img_rgb):
    # Bilder skalieren
    img_rgb.shape[1]
    img_width = 800
    scale =img_width / img_rgb.shape[1]

    skaliertes_img = cv2.resize(img_rgb, (0, 0), fx=scale, fy=scale)
    return skaliertes_img

#### Bilateral Filter (Vorlesung 4 Folie 14–25)

Kanten bleiben erhalten. Sinnvoll für Schilder

let src = cv.imread('canvasInput');  
let dst = new cv.Mat();  
cv.cvtColor(src, src, cv.COLOR_RGBA2RGB, 0);  
// You can try more different parameters  
cv.bilateralFilter(src, dst, 9, 75, 75, cv.BORDER_DEFAULT);  
cv.imshow('canvasOutput', dst);  
src.delete(); dst.delete();  

In [ ]:
def apply_bilateral_filter(img_rgb):
    # Biletaral Filter 

    img_filtered = cv2.bilateralFilter(img_rgb, d=9, sigmaColor=75, sigmaSpace=75)
    return img_filtered

#### CLAHE Vorlesung 6

In [ ]:
def apply_clahe(img_filtered):
    #Vorbvereitung
    #rgb zu hsv (HE wird auf V-Kanal angewendet)
    image_hsv_raw = cv2.cvtColor(img_filtered, cv2.COLOR_RGB2HSV)
    #print(image_hsv_raw.shape)
    v_channel = image_hsv_raw[:, :, 2]
    #print(v_channel.shape)

    # CLAHE erstellen und auf v-kanal
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    v_clahe = clahe.apply(v_channel)
    return v_clahe

---

In [ ]:
def convert_to_hsv(img_filtered):
    # rgb zu hsv
    img_hsv = cv2.cvtColor(img_filtered, cv2.COLOR_RGB2HSV)
    return img_hsv

In [ ]:
def create_color_masks(img_hsv):
    # Maske/ Farbräume für das HSV Treshholding
    masks = {
        "rot":   cv2.inRange(img_hsv,   np.array([0, 125, 50]), np.array([10, 255, 250])) 
               | cv2.inRange(img_hsv,   np.array([170, 125, 50]), np.array([179, 255, 250])),
        "blau": cv2.inRange(img_hsv, np.array([90, 80, 50]),  np.array([120, 255, 255])),
        "gelb": cv2.inRange(img_hsv, np.array([20, 120, 130]),  np.array([25, 220, 200])),

    }
    return masks

Mathematische Morphologie  
Vorlesung 5  
Seite 11  
  
import cv2  
import numpy as np  
  
img = cv2.imread('j.png',0)  
kernel = np.ones((5,5),np.uint8)  
erosion = cv2.erode(img, kernel, iterations = 1)  

  
##### Rectangular Kernel  
 cv2.getStructuringElement(cv2.MORPH_RECT, (5,5))  
array([[1, 1, 1, 1, 1],  
[1, 1, 1, 1, 1],  
[1, 1, 1, 1, 1],  
[1, 1, 1, 1, 1],  
[1, 1, 1, 1, 1]], dtype=uint8)  
  
##### Elliptical Kernel  
 cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))  
array([[0, 0, 1, 0, 0],  
[1, 1, 1, 1, 1],  
[1, 1, 1, 1, 1],  
[1, 1, 1, 1, 1],  
[0, 0, 1, 0, 0]], dtype=uint8)  
  
##### Cross-shaped Kernel  
 cv2.getStructuringElement(cv2.MORPH_CROSS, (5,5))  
array([[0, 0, 1, 0, 0],  
[0, 0,1, 0, 0],  
[1, 1, 1, 1, 1],  
[0, 0,1, 0,0],  
[0, 0, 1, 0, 0]], dtype=uint8)  

Seite 28  
opening = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel)  
  
Seite 29  
closing = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)  

In [ ]:
def clean_color_masks(masks):
    kernel_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (4, 4))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    mask_rot_clean  = cv2.morphologyEx(masks["rot"],  cv2.MORPH_OPEN,  kernel_open,  iterations=1)
    mask_rot_clean  = cv2.morphologyEx(mask_rot_clean, cv2.MORPH_CLOSE, kernel_close, iterations=2)

    mask_blau_clean = cv2.morphologyEx(masks["blau"], cv2.MORPH_OPEN,  kernel_open,  iterations=1)
    mask_blau_clean = cv2.morphologyEx(mask_blau_clean, cv2.MORPH_CLOSE, kernel_close, iterations=2)

    mask_gelb_clean = cv2.morphologyEx(masks["gelb"], cv2.MORPH_OPEN,  kernel_open,  iterations=1)
    mask_gelb_clean = cv2.morphologyEx(mask_gelb_clean, cv2.MORPH_CLOSE, kernel_close, iterations=2)

    clean_masks = {
        "rot": mask_rot_clean,
        "blau": mask_blau_clean,
        "gelb": mask_gelb_clean
    }
    return clean_masks

Vorlesung 2 Seite 49
  
import cv2  
from skimage import img_as_ubyte  
cv_blobs = img_as_ubyte(blobs)  
num_labels, labels_im = cv2.connectedComponents(cv_blobs)  
print("Opencv components:", num_labels)  
plt.imshow(labels_im, cmap='gray')  

In [ ]:
MIN_AREA = 600

Minimal größer von Komonenten verlangen um kleien übrig gebliebene fetzen nicht zu segmentieren  
Aspect Ratio nutzten, um Rechtecke die keine Schilder sein können, nicht segmentiert werden

In [ ]:
def find_signs(mask, mask_color, connectivity=8):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)

    sings = []

    for label_id in range(1, num_labels):
        x = stats[label_id, cv2.CC_STAT_LEFT]
        y = stats[label_id, cv2.CC_STAT_TOP]
        w = stats[label_id, cv2.CC_STAT_WIDTH]
        h = stats[label_id, cv2.CC_STAT_HEIGHT]
        area = stats[label_id, cv2.CC_STAT_AREA]
        cx, cy = centroids[label_id]
        aspect_ratio = w / h

        if area < MIN_AREA or not (0.5 <= aspect_ratio <= 2.0):
            continue
        
        #Convex Hull (Vorlesung 5, Folie 53)
        region_mask = (labels == label_id).astype(np.uint8)
        contours, _ = cv2.findContours(region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        hull = cv2.convexHull(contours[0])
        hull_area = cv2.contourArea(hull)
        if hull_area > 0:
            hull_ratio = area / hull_area
        else:
            hull_ratio = 0.0

        sings.append(
            {"color": mask_color,
            "bbox": (x, y, w, h), 
            "area": area, 
            "centroid": (cx, cy), 
            "aspect_ratio": aspect_ratio,
            "hull_ratio": hull_ratio,}
            )
    return sings

In [ ]:
def detect_signs(clean_masks):
    # Alle Schilder in den 3 Masken finden
    found_signs = []

    for color, mask in clean_masks.items():
        signs = find_signs(mask, color)
        found_signs += signs

    print("Found signs:", len(found_signs))
    for sign in found_signs:
        print(sign)
    return found_signs

https://docs.python.org/3/howto/sorting.html  
Array sortieren:  
  
student_tuples = [  
    ('john', 'A', 15),  
    ('jane', 'B', 12),  
    ('dave', 'B', 10),  
]  
sorted(student_tuples, key=lambda student: student[2])   # sort by age  
[('dave', 'B', 10), ('jane', 'B', 12), ('john', 'A', 15)]  

In [ ]:
def filter_overlapping_signs(found_signs):
    # Entferne kleiner Bounding Boxes die in einer großen Liegen
    found_signs = sorted(found_signs, key=lambda sign: sign["area"], reverse=True)


    filtered_signs = []

    for sign in found_signs:
        x_begin, y_begin, w, h = sign["bbox"]
        x_end, y_end = x_begin + w, y_begin + h
        
        is_inside_bigger = False  
        
        for bigger in filtered_signs:
            x_box_begin, y_box_begin, w_box, h_box = bigger["bbox"]
            x_box_end, y_box_end = x_box_begin + w_box, y_box_begin + h_box
            
            if (x_begin >= x_box_begin and x_end <= x_box_end and 
                y_begin >= y_box_begin and y_end <= y_box_end):
                is_inside_bigger = True
                break
        
        if not is_inside_bigger:
            filtered_signs.append(sign)
            
    found_signs = filtered_signs

    print("Nach entfernen von überflüssigen Bounding Boxen:", len(found_signs), "Schilder")
    for sign in found_signs:
        print(sign)
    return found_signs

In [ ]:
#Klassifikation der Schilder anhand Form und Farbe

def classify_sign(color, area, w, h, cx, cy, x, y, hull_ratio):
    fill_ratio = area / (w * h)            # Fläche / Bbox Fläche
    aspect_ratio = w / h
    is_squarish = 0.75 <= aspect_ratio <= 1.25
    is_wide = aspect_ratio > 1.5
    '''
    if hull_ratio < 0.70:
        return "Unbekannt"
    '''
    cy_rel = (cy - y) / h

    # Rote Schilder
    if color == "rot":
        if not is_squarish:
            return "Unbekannt"
        if fill_ratio > 0.30:
            return "Verbotsschild"   
        if cy_rel > 0.55:
            return "Gefahrenschild"
        if cy_rel < 0.45:
            return "Vorfahrt gewaehren"
        return "Unbekannt"
    
    # Blaue Schilder
    if color == "blau":
        if is_squarish:
            return "Gebotsschild"              
        return "Hinweisschild"                 
    
    # Gelbe Schilder
    if color == "gelb":
        if fill_ratio < 0.65 and is_squarish:
            return "Vorfahrtsstrasse"           
        if fill_ratio > 0.65 and is_wide:
            return "Ortsschild / Wegweiser"
        return "Unbekannt"

    return "Unbekannt"

In [ ]:
def classify_found_signs(found_signs):
    # Klassifikation der gefundenen Schilder
    for sign in found_signs:
        x, y, w, h = sign["bbox"]
        cx, cy = sign["centroid"]

        sign["klasse"] = classify_sign(
            sign["color"], 
            sign["area"], 
            w, h, cx, cy, x, y,
            sign["hull_ratio"]
        )

    print("Klassifizierte Schilder:", len(found_signs))
    for sign in found_signs:
        print(sign)
    return found_signs

In [ ]:
classify_images(IMAGE_PATH)